# 02 分析用データの作成

## 目的

複数のデータセットから、歩行能力（WalkingAbility）の分析・予測に
使用できるデータセットを作成する。

### 方針

- 1行を「1個体 × 1観測時点」とする
- 個体・日齢・Penなどを用いてデータを対応付ける
- 行動データを横持ち形式に変換する
- 加速度特徴量と福祉指標を統合する
- 欠損値や重複データを確認する

最終的に、WalkingAbilityを目的変数とした分析に使用できる形式に整える。

In [2]:
import pandas as pd
import numpy as np

In [3]:
behaviour_df = pd.read_excel(
    "../data/raw/behaviour_welfare.xlsx"
)

print("データサイズ:", behaviour_df.shape)
display(behaviour_df.head())

データサイズ: (4590, 24)


,Bird_Pen_Strain_Date_Time,Bird,Pen,Strain,Age_Days,Tiers,Total_Duration_Sec,Frequency,Average_BoutLength,SexFM,...,BreastplumageDirtiness,Pododermatitis,HockBurn,LegStraightnessLeft,LegStraightnessRight,PerTimeAct,Pen_Bird_Strain_Date,Pen_Bird_Strain_Date_Accel,Date,Time
0,1_11_NB_03.04.2019_11.40-12.20,1,11,NB,48,Drink_Standing,0.0,0,0.0,M,...,0,0,0.5,B1,B1,18.135498,11_1_NB_03.04,Pen11_1_NB_03.04.2019_46163,03.04.2019,11.40-12.20
1,1_11_NB_03.04.2019_11.40-12.20,1,11,NB,48,Self-Peck_Mark,0.0,0,0.0,M,...,0,0,0.5,B1,B1,18.135498,11_1_NB_03.04,Pen11_1_NB_03.04.2019_46163,03.04.2019,11.40-12.20
2,1_11_NB_03.04.2019_11.40-12.20,1,11,NB,48,Running,0.0,0,0.0,M,...,0,0,0.5,B1,B1,18.135498,11_1_NB_03.04,Pen11_1_NB_03.04.2019_46163,03.04.2019,11.40-12.20
3,1_11_NB_03.04.2019_11.40-12.20,1,11,NB,48,Sidelying_Inactive,0.0,0,0.0,M,...,0,0,0.5,B1,B1,18.135498,11_1_NB_03.04,Pen11_1_NB_03.04.2019_46163,03.04.2019,11.40-12.20
4,1_11_NB_03.04.2019_11.40-12.20,1,11,NB,48,Wing_flap,0.0,0,0.0,M,...,0,0,0.5,B1,B1,18.135498,11_1_NB_03.04,Pen11_1_NB_03.04.2019_46163,03.04.2019,11.40-12.20


## 行動データの横持ち変換

behaviour_welfareでは、1つの観測について行動の種類（Tiers）ごとに
別々の行として記録されている。

分析や機械学習で扱いやすくするため、
1行を「1個体 × 1観測時点」とし、
各行動の継続時間（Total_Duration_Sec）を列として展開する。

In [4]:
behaviour_wide = behaviour_df.pivot_table(
    index=[
        "Pen_Bird_Strain_Date",
        "Bird",
        "Pen",
        "Strain",
        "Age_Days",
        "SexFM",
        "WalkingAbility"
    ],
    columns="Tiers",
    values="Total_Duration_Sec",
    aggfunc="mean"
).reset_index()

# Tiersが列名として残るので解除
behaviour_wide.columns.name = None

print("変換前:", behaviour_df.shape)
print("変換後:", behaviour_wide.shape)

display(behaviour_wide.head())

変換前: (4590, 24)
変換後: (135, 41)


,Pen_Bird_Strain_Date,Bird,Pen,Strain,Age_Days,SexFM,WalkingAbility,Crawling,Drink_Standing,Dust_Bathing,...,Running,Self-Peck_Accel,Self-Peck_Mark,Shuffle,Sidelying_Inactive,Sit_Inactive,Stand_Inactive,Walking,Wing_assisted_run,Wing_flap
0,11_1_NB_03.04,1,11,NB,48,M,1,2.41,0.00,0.0,...,0.0,0.00,0.0,6.25,0.0,311.585,36.78,25.470,0.0,0.0
1,11_1_NB_12.03,1,11,NB,26,M,1,0.00,0.00,0.0,...,0.0,0.00,0.0,11.81,0.0,27.300,19.26,24.310,0.0,0.0
2,11_2_NB_03.04,2,11,NB,48,M,1,0.00,0.00,0.0,...,0.0,0.61,0.0,18.30,0.0,288.200,29.48,33.320,0.0,0.0
3,11_2_NB_12.03,2,11,NB,26,M,1,0.00,0.00,0.0,...,0.0,0.00,0.0,22.13,0.0,29.070,29.28,35.900,0.0,0.0
4,11_2_NB_26.03,2,11,NB,40,M,1,0.00,18.03,0.0,...,0.0,0.00,0.0,25.95,0.0,63.825,50.22,78.275,0.0,0.0


## 横持ち変換後のデータ確認

横持ち変換によって、135観測 × 41列のデータとなった。
変換後のデータについて、欠損値と重複データの有無を確認する。

In [5]:
# データサイズ
print("データサイズ:", behaviour_wide.shape)

# 重複確認
print("完全重複行:", behaviour_wide.duplicated().sum())

# 観測IDの重複確認
print(
    "観測IDの重複:",
    behaviour_wide["Pen_Bird_Strain_Date"].duplicated().sum()
)

# 欠損値がある列だけ表示
missing = behaviour_wide.isnull().sum()
display(missing[missing > 0].sort_values(ascending=False))

データサイズ: (135, 41)
完全重複行: 0
観測IDの重複: 0


Series([], dtype: int64)

### 確認結果

- 横持ち変換後は135観測、41列となった
- 完全に重複する行は存在しなかった
- Pen_Bird_Strain_Date は全135観測で一意だった
- 欠損値は確認されなかった

以上から、行動データは1観測1行の形式に正常に変換できたと判断した。

## 加速度・歩行能力データの前処理

walking_welfare.csv を読み込み、
行動データと統合するために観測単位や識別情報を確認する。

In [6]:
walking_df = pd.read_csv(
    "../data/raw/walking_welfare.csv"
)

print("データサイズ:", walking_df.shape)

display(walking_df.head())

print("\n列数:", len(walking_df.columns))

データサイズ: (2497, 120)


,Pen_Bird_Strain_Date,Age_Days,SexFM,Repeated_Weightkg,Welfare_Assess_weightkg,WalkingAbility,Feathercover,BreastplumageDirtiness,Pododermatitis,HockBurn,...,Z_Static_filt.Min,Z_Static_filt.Max,Z_Static_filt.AbsMean,Z_Static_filt.IQR,Z_Static_filt.Skewness,Z_Static_filt.Kurtosis,Z_Static_filt.Entropy,Z_Static_filt.Peak_Trough.x,Z_Static_filt.Peak_Trough.y,UniqueBirds
0,11_4_NB_03.04.2019,48,M,2.078,2.491,1,0,0,0,0.5,...,-1.001594,-0.995931,0.999195,0.002326,0.525968,2.206204,5.703781,73.60555,-0.995930,Pen11_4_NB
1,11_4_NB_03.04.2019,48,M,2.078,2.491,1,0,0,0,0.5,...,-0.999236,-0.989803,0.994124,0.002721,-0.157488,2.480772,5.703779,38.75735,-0.989803,Pen11_4_NB
2,11_4_NB_03.04.2019,48,M,2.078,2.491,1,0,0,0,0.5,...,-1.000490,-0.989829,0.995626,0.005538,0.366600,1.888628,5.703777,296.67720,-0.989664,Pen11_4_NB
3,11_4_NB_03.04.2019,48,M,2.078,2.491,1,0,0,0,0.5,...,-0.999994,-0.991225,0.997022,0.001756,0.762005,3.187205,5.703781,47.47332,-0.990505,Pen11_4_NB
4,11_4_NB_03.04.2019,48,M,2.078,2.491,1,0,0,0,0.5,...,-0.999343,-0.992396,0.995572,0.003539,-0.267254,1.868994,5.703780,103.83390,-0.992396,Pen11_4_NB



列数: 120


### 観測単位の確認

walking_welfareでは同じPen_Bird_Strain_Dateが複数行存在するため、
各観測に含まれる加速度データの行数を確認する。

In [7]:
# 観測IDの種類数
print(
    "観測ID数:",
    walking_df["Pen_Bird_Strain_Date"].nunique()
)

# 1観測あたりの行数
records_per_obs = (
    walking_df["Pen_Bird_Strain_Date"]
    .value_counts()
)

display(records_per_obs.describe())

display(records_per_obs.head(10))

観測ID数: 15


count      15.000000
mean      166.466667
std       256.233116
min        28.000000
25%        64.000000
50%       107.000000
75%       148.000000
max      1072.000000
Name: count, dtype: float64

Pen_Bird_Strain_Date
52_OO_R_13.03.2019     1072
12_BB_R_12.03.2019      212
11_4_NB_03.04.2019      185
51_3_NB_27.03.2019      175
21_PuB_H_03.04.2019     121
51_3_NB_03.04.2019      109
51_3_NB_16.03.2019      109
22_1_NB_26.03.2019      107
11_4_NB_12.03.2019       89
22_1_NB_12.03.2019       67
Name: count, dtype: int64

In [8]:
behaviour_ids = set(behaviour_wide["Pen_Bird_Strain_Date"])
walking_ids = set(walking_df["Pen_Bird_Strain_Date"])

print("behaviour観測ID数:", len(behaviour_ids))
print("walking観測ID数:", len(walking_ids))
print("共通する観測ID数:", len(behaviour_ids & walking_ids))
print("behaviour側だけ:", len(behaviour_ids - walking_ids))
print("walking側だけ:", len(walking_ids - behaviour_ids))

behaviour観測ID数: 135
walking観測ID数: 15
共通する観測ID数: 0
behaviour側だけ: 135
walking側だけ: 15


In [9]:
walking_df["Observation_ID"] = (
    walking_df["Pen_Bird_Strain_Date"]
    .str.replace(".2019", "", regex=False)
)

behaviour_wide["Observation_ID"] = (
    behaviour_wide["Pen_Bird_Strain_Date"]
)

behaviour_ids = set(behaviour_wide["Observation_ID"])
walking_ids = set(walking_df["Observation_ID"])

print("behaviour観測ID数:", len(behaviour_ids))
print("walking観測ID数:", len(walking_ids))
print("共通する観測ID数:", len(behaviour_ids & walking_ids))

display(sorted(behaviour_ids & walking_ids))

behaviour観測ID数: 135
walking観測ID数: 15
共通する観測ID数: 15


C:\Users\81808\AppData\Local\Temp\ipykernel_8140\728839890.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  walking_df["Observation_ID"] = (


['11_4_NB_03.04',
 '11_4_NB_12.03',
 '12_BB_R_12.03',
 '21_OO_H_03.04',
 '21_OO_H_12.03',
 '21_OO_H_26.03',
 '21_PuB_H_03.04',
 '21_PuB_H_26.03',
 '22_1_NB_03.04',
 '22_1_NB_12.03',
 '22_1_NB_26.03',
 '51_3_NB_03.04',
 '51_3_NB_16.03',
 '51_3_NB_27.03',
 '52_OO_R_13.03']

In [10]:
# 数値列を確認
numeric_cols = walking_df.select_dtypes(include="number").columns.tolist()

print("数値列数:", len(numeric_cols))
print(numeric_cols)

数値列数: 111
['Age_Days', 'Repeated_Weightkg', 'Welfare_Assess_weightkg', 'WalkingAbility', 'Feathercover', 'BreastplumageDirtiness', 'Pododermatitis', 'HockBurn', 'Time.window', 'Prop.behaviour', 'MostFreq.Behaviour_1Si_2Sta_3Wa', 'Accel', 'X_Dynamic_Freq.Min', 'X_Dynamic_Freq.Max', 'X_Dynamic_Freq.AbsMean', 'X_Dynamic_Freq.IQR', 'X_Dynamic_Amp.Min', 'X_Dynamic_Amp.Max', 'X_Dynamic_Amp.AbsMean', 'X_Dynamic_Amp.IQR', 'Y_Dynamic_Freq.Min', 'Y_Dynamic_Freq.Max', 'Y_Dynamic_Freq.AbsMean', 'Y_Dynamic_Freq.IQR', 'Y_Dynamic_Amp.Min', 'Y_Dynamic_Amp.Max', 'Y_Dynamic_Amp.AbsMean', 'Y_Dynamic_Amp.IQR', 'Z_Dynamic_Freq.Min', 'Z_Dynamic_Freq.Max', 'Z_Dynamic_Freq.AbsMean', 'Z_Dynamic_Freq.IQR', 'Z_Dynamic_Amp.Min', 'Z_Dynamic_Amp.Max', 'Z_Dynamic_Amp.AbsMean', 'Z_Dynamic_Amp.IQR', 'Roll.Min', 'Roll.Max', 'Roll.AbsMean', 'Roll.IQR', 'Pitch.Min', 'Pitch.Max', 'Pitch.AbsMean', 'Pitch.IQR', 'Yaw.Min', 'Yaw.Max', 'Yaw.AbsMean', 'Yaw.IQR', 'X_Dynamic_filt.Min', 'X_Dynamic_filt.Max', 'X_Dynamic_filt.AbsMea

In [11]:
# 加速度由来の特徴量だけを抽出
feature_prefixes = (
    "X_Dynamic",
    "Y_Dynamic",
    "Z_Dynamic",
    "Roll",
    "Pitch",
    "Yaw",
    "SVM",
    "X_Static",
    "Y_Static",
    "Z_Static"
)

accel_cols = [
    col for col in walking_df.columns
    if col.startswith(feature_prefixes)
]

print("加速度特徴量数:", len(accel_cols))
print(accel_cols)

加速度特徴量数: 99
['X_Dynamic_Freq.Min', 'X_Dynamic_Freq.Max', 'X_Dynamic_Freq.AbsMean', 'X_Dynamic_Freq.IQR', 'X_Dynamic_Amp.Min', 'X_Dynamic_Amp.Max', 'X_Dynamic_Amp.AbsMean', 'X_Dynamic_Amp.IQR', 'Y_Dynamic_Freq.Min', 'Y_Dynamic_Freq.Max', 'Y_Dynamic_Freq.AbsMean', 'Y_Dynamic_Freq.IQR', 'Y_Dynamic_Amp.Min', 'Y_Dynamic_Amp.Max', 'Y_Dynamic_Amp.AbsMean', 'Y_Dynamic_Amp.IQR', 'Z_Dynamic_Freq.Min', 'Z_Dynamic_Freq.Max', 'Z_Dynamic_Freq.AbsMean', 'Z_Dynamic_Freq.IQR', 'Z_Dynamic_Amp.Min', 'Z_Dynamic_Amp.Max', 'Z_Dynamic_Amp.AbsMean', 'Z_Dynamic_Amp.IQR', 'Roll.Min', 'Roll.Max', 'Roll.AbsMean', 'Roll.IQR', 'Pitch.Min', 'Pitch.Max', 'Pitch.AbsMean', 'Pitch.IQR', 'Yaw.Min', 'Yaw.Max', 'Yaw.AbsMean', 'Yaw.IQR', 'X_Dynamic_filt.Min', 'X_Dynamic_filt.Max', 'X_Dynamic_filt.AbsMean', 'X_Dynamic_filt.IQR', 'X_Dynamic_filt.Skewness', 'X_Dynamic_filt.Kurtosis', 'X_Dynamic_filt.Entropy', 'X_Dynamic_filt.Peak_Trough.x', 'X_Dynamic_filt.Peak_Trough.y', 'Y_Dynamic_filt.Min', 'Y_Dynamic_filt.Max', 'Y_Dynamic_

In [12]:
# walkingデータの1行が何を表しているか確認

check_cols = [
    "Observation_ID",
    "Time.window",
    "MostFreq.Behaviour_1Si_2Sta_3Wa",
    "Prop.behaviour",
    "Accel"
]

display(walking_df[check_cols].head(20))

print("Time.window のユニーク数:",
      walking_df["Time.window"].nunique())

print("\nMostFreq.Behaviour の種類:")
display(
    walking_df["MostFreq.Behaviour_1Si_2Sta_3Wa"]
    .value_counts(dropna=False)
)

print("\nAccel の種類:")
display(
    walking_df["Accel"]
    .value_counts(dropna=False)
)

,Observation_ID,Time.window,MostFreq.Behaviour_1Si_2Sta_3Wa,Prop.behaviour,Accel
0,11_4_NB_03.04,100,1,1.000,46167
1,11_4_NB_03.04,1000,1,1.000,46167
2,11_4_NB_03.04,1001,1,1.000,46167
3,11_4_NB_03.04,1002,1,1.000,46167
4,11_4_NB_03.04,1003,1,0.697,46167
5,11_4_NB_03.04,1005,1,0.670,46167
6,11_4_NB_03.04,1006,1,1.000,46167
7,11_4_NB_03.04,1007,1,1.000,46167
8,11_4_NB_03.04,1008,1,1.000,46167
9,11_4_NB_03.04,1009,1,1.000,46167


Time.window のユニーク数: 1463

MostFreq.Behaviour の種類:


MostFreq.Behaviour_1Si_2Sta_3Wa
1    1865
2     333
3     299
Name: count, dtype: int64


Accel の種類:


Accel
46422    1072
46152     393
46749     237
46298     212
44829     186
46167     185
46410     123
46163      89
Name: count, dtype: int64

## 加速度データの構造確認

- walkingデータは2497行、120列
- 加速度由来の特徴量は99列
- behaviourデータ135観測のうち、walkingデータが存在するのは15観測
- 観測IDの日付表記（`.2019`）を統一すると、walking側15観測すべてがbehaviour側と一致した
- walkingデータは1観測につき複数の時間窓（Time.window）を持つ
- MostFreq.Behaviour は3分類
  - 1: Sitting
  - 2: Standing
  - 3: Walking
- 全2497行の内訳
  - Sitting: 1865
  - Standing: 333
  - Walking: 299
- 今後は時間窓単位の加速度特徴量を利用し、WalkingAbilityとの関係を分析する

In [13]:
# Walkingの時間窓だけ抽出
walking_only = walking_df[
    walking_df["MostFreq.Behaviour_1Si_2Sta_3Wa"] == 3
].copy()

print("Walkingの行数:", len(walking_only))
print("Walkingがある観測数:", walking_only["Observation_ID"].nunique())

# 観測ごとのWalking時間窓数
display(
    walking_only["Observation_ID"]
    .value_counts()
    .sort_index()
)

Walkingの行数: 299
Walkingがある観測数: 14


Observation_ID
11_4_NB_03.04     13
11_4_NB_12.03     23
12_BB_R_12.03      5
21_OO_H_03.04     23
21_OO_H_12.03     27
21_OO_H_26.03     24
21_PuB_H_03.04    26
22_1_NB_03.04     12
22_1_NB_12.03     27
22_1_NB_26.03     24
51_3_NB_03.04     21
51_3_NB_16.03     29
51_3_NB_27.03     16
52_OO_R_13.03     29
Name: count, dtype: int64

In [14]:
# Walkingが存在しない観測IDを確認
all_ids = set(walking_df["Observation_ID"])
walking_ids = set(walking_only["Observation_ID"])

no_walking_ids = all_ids - walking_ids

print("Walkingがない観測ID:", no_walking_ids)

Walkingがない観測ID: {'21_PuB_H_26.03'}


In [15]:
# Walkingデータの加速度特徴量を観測単位で平均化

walking_accel_mean = (
    walking_only
    .groupby("Observation_ID")[accel_cols]
    .mean()
    .reset_index()
)

print("集約後のデータサイズ:", walking_accel_mean.shape)
display(walking_accel_mean.head())

集約後のデータサイズ: (14, 100)


,Observation_ID,X_Dynamic_Freq.Min,X_Dynamic_Freq.Max,X_Dynamic_Freq.AbsMean,X_Dynamic_Freq.IQR,X_Dynamic_Amp.Min,X_Dynamic_Amp.Max,X_Dynamic_Amp.AbsMean,X_Dynamic_Amp.IQR,Y_Dynamic_Freq.Min,...,Y_Static_filt.Peak_Trough.y,Z_Static_filt.Min,Z_Static_filt.Max,Z_Static_filt.AbsMean,Z_Static_filt.IQR,Z_Static_filt.Skewness,Z_Static_filt.Kurtosis,Z_Static_filt.Entropy,Z_Static_filt.Peak_Trough.x,Z_Static_filt.Peak_Trough.y
0,11_4_NB_03.04,0.001051,0.048974,0.024727,0.023615,0.000515,0.284558,0.027344,0.020816,0.001077,...,0.171737,-1.082306,-1.007160,1.051431,0.025700,0.447392,2.600404,5.703547,159.622405,-1.006637
1,11_4_NB_12.03,0.001189,0.049014,0.024766,0.023685,0.002090,0.477719,0.061802,0.069582,0.001102,...,0.118125,-1.068411,-0.857336,1.005759,0.047855,0.687088,3.842122,5.694262,146.063662,-0.846432
2,12_BB_R_12.03,0.001250,0.048750,0.024565,0.024031,0.000151,0.070775,0.007634,0.007825,0.001063,...,0.099258,-0.993883,-0.908002,0.957737,0.035962,0.619064,2.663877,5.708719,123.140922,-0.903687
3,21_OO_H_03.04,0.001015,0.048899,0.024565,0.024014,0.001559,0.528781,0.047490,0.034830,0.001102,...,0.120223,-1.074480,-0.901428,1.027542,0.033554,0.517441,3.056917,5.706444,128.814738,-0.898507
4,21_OO_H_12.03,0.001161,0.048951,0.024712,0.024191,0.000425,0.199725,0.021543,0.019314,0.001198,...,0.020789,-1.067652,-0.898928,1.006572,0.055158,0.586681,3.091001,5.692691,155.919676,-0.896799


In [ ]:
# 同一観測内でラベル情報が一意であることを確認
label_check = (
    walking_df
    .groupby("Observation_ID")[
        ["Age_Days", "SexFM", "WalkingAbility"]
    ]
    .nunique()
)

print("複数の値を持つ観測数:")
print((label_check > 1).any(axis=1).sum())

# 各観測のWalkingAbilityなどの基本情報を取得
walking_labels = (
    walking_df[
        ["Observation_ID", "Age_Days", "SexFM", "WalkingAbility"]
    ]
    .drop_duplicates(subset="Observation_ID")
)

print("ラベルデータ:", walking_labels.shape)
display(walking_labels)

複数の値を持つ観測数:
0
ラベルデータ: (15, 4)


,Observation_ID,Age_Days,SexFM,WalkingAbility
0,11_4_NB_03.04,48,M,1
185,11_4_NB_12.03,26,M,1
274,12_BB_R_12.03,26,M,2
486,21_OO_H_03.04,48,F,1
514,21_OO_H_12.03,26,F,1
569,21_OO_H_26.03,40,F,1
609,21_PuB_H_03.04,48,F,1
730,21_PuB_H_26.03,40,F,1
795,22_1_NB_03.04,48,F,1
858,22_1_NB_12.03,26,F,1


In [17]:
# 加速度特徴量とラベルを結合
model_df = walking_accel_mean.merge(
    walking_labels,
    on="Observation_ID",
    how="left"
)

print("結合後のデータサイズ:", model_df.shape)

# WalkingAbilityの内訳
print("\nWalkingAbilityの内訳:")
print(model_df["WalkingAbility"].value_counts())

display(model_df.head())

結合後のデータサイズ: (14, 103)

WalkingAbilityの内訳:
WalkingAbility
1    12
2     2
Name: count, dtype: int64


,Observation_ID,X_Dynamic_Freq.Min,X_Dynamic_Freq.Max,X_Dynamic_Freq.AbsMean,X_Dynamic_Freq.IQR,X_Dynamic_Amp.Min,X_Dynamic_Amp.Max,X_Dynamic_Amp.AbsMean,X_Dynamic_Amp.IQR,Y_Dynamic_Freq.Min,...,Z_Static_filt.AbsMean,Z_Static_filt.IQR,Z_Static_filt.Skewness,Z_Static_filt.Kurtosis,Z_Static_filt.Entropy,Z_Static_filt.Peak_Trough.x,Z_Static_filt.Peak_Trough.y,Age_Days,SexFM,WalkingAbility
0,11_4_NB_03.04,0.001051,0.048974,0.024727,0.023615,0.000515,0.284558,0.027344,0.020816,0.001077,...,1.051431,0.025700,0.447392,2.600404,5.703547,159.622405,-1.006637,48,M,1
1,11_4_NB_12.03,0.001189,0.049014,0.024766,0.023685,0.002090,0.477719,0.061802,0.069582,0.001102,...,1.005759,0.047855,0.687088,3.842122,5.694262,146.063662,-0.846432,26,M,1
2,12_BB_R_12.03,0.001250,0.048750,0.024565,0.024031,0.000151,0.070775,0.007634,0.007825,0.001063,...,0.957737,0.035962,0.619064,2.663877,5.708719,123.140922,-0.903687,26,M,2
3,21_OO_H_03.04,0.001015,0.048899,0.024565,0.024014,0.001559,0.528781,0.047490,0.034830,0.001102,...,1.027542,0.033554,0.517441,3.056917,5.706444,128.814738,-0.898507,48,F,1
4,21_OO_H_12.03,0.001161,0.048951,0.024712,0.024191,0.000425,0.199725,0.021543,0.019314,0.001198,...,1.006572,0.055158,0.586681,3.091001,5.692691,155.919676,-0.896799,26,F,1


In [18]:
# 最終データの欠損確認
missing = model_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("欠損がある列数:", len(missing))
display(missing)

欠損がある列数: 0


Series([], dtype: int64)

In [19]:
# 前処理済みデータを保存
model_df.to_csv(
    "../data/processed/walking_model_data.csv",
    index=False
)

print("保存完了")

保存完了


## 前処理結果

WalkingAbilityは歩行能力を示す指標であるため、
歩行時の加速度特徴に着目する目的で、
MostFreq.BehaviourがWalking（3）の時間窓のみを分析対象とした。

Walkingとして分類された時間窓のみを抽出し、
加速度特徴量99項目を観測単位で平均化した。

- 元の加速度データ：2,497行
- Walkingデータ：299行
- Walkingを含む観測：14観測
- 加速度特徴量：99項目
- 最終データ：14行 × 103列
- WalkingAbility
  - 1：12観測
  - 2：2観測
- 欠損値：なし

Walkingが存在しなかった `21_PuB_H_26.03` は、
Walkingの加速度特徴量を作成できないため分析対象外となった。

前処理済みデータを
`data/processed/walking_model_data.csv`
として保存した。